# CAT-3 — Git-like branching on Nessie (SQL-first)

Nessie is the **only** catalog in this repo that exposes real Git-style semantics on your **whole warehouse**: `CREATE BRANCH`, `USE REFERENCE`, `MERGE BRANCH`. Same three moves you make dozens of times a day in code, applied to entire datasets.

The narrative is a common one — an **ETL rehearsal**. You need to develop a new `staging.orders` table without touching production. On a filesystem-backed catalog (Hadoop / Hive) you'd copy the whole warehouse; on Nessie it's one metadata-only commit that costs one row in Nessie's ref store and zero bytes on S3.

**Break → Do → Detect → Prove:**
1. **Do**: `CREATE BRANCH dev FROM main`, switch onto it, write a table, verify rows.
2. **Detect** (isolation): switch back to `main` — the table doesn't exist there yet.
3. **Fix/Merge**: `MERGE BRANCH dev INTO main` — one atomic commit.
4. **Prove**: on `main`, the rows are now visible.

> **Prereqs:** `make up && make catalogs-up`. This uses the Nessie Spark SQL extension (`nessie-spark-extensions-4.0_2.13:0.108.0` — baked into the image) so branching happens **in SQL**, not curl. The parallel control-plane surface (raw Nessie REST) lives in [`../catalog_api_playground.ipynb`](../catalog_api_playground.ipynb).

**Companion:** [CAT-1](./cat1_nessie_intro.ipynb) already showed *where the pointer lives* (a REST service, not `version-hint.text`). This lesson uses that same pointer to open a **second, parallel history** and then merges it back.

In [1]:
from common.spark_session import spark

MAIN_TABLE = "nessie_catalog.staging.orders"   # the table we'll build
DEV_BRANCH = "cat3_dev"                          # branch name (unique so peer runs don't clash)

# Idempotent reset — whichever branch is currently active, hop back to main,
# drop the table if it exists, and remove any leftover dev branch. DROP TABLE
# on Iceberg tries to read the current metadata.json before deleting; if a
# prior run left a stale pointer whose S3 files were wiped (e.g. `make clean`),
# that read 404s. Tolerated — the pointer is overwritten by our fresh CREATE.
try:
    spark.sql("USE REFERENCE main IN nessie_catalog").collect()
except Exception:
    pass
try:
    spark.sql(f"DROP TABLE IF EXISTS {MAIN_TABLE}").collect()
except Exception as e:
    print(f"tolerated stale DROP TABLE: {type(e).__name__}: {str(e)[:100]}")
try:
    spark.sql(f"DROP BRANCH {DEV_BRANCH} IN nessie_catalog").collect()
except Exception:
    pass  # not there — fine

spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie_catalog.staging").collect()
print(f"reset done — clean slate on nessie_catalog / main")

reset done — clean slate on nessie_catalog / main


## 1. Fork a branch from `main`

`CREATE BRANCH ... FROM main` is one metadata-only commit in Nessie's ref store. `USE REFERENCE` re-targets **this same** `nessie_catalog` handle at the new branch — no second catalog to register, no config to reload.

This is the difference from an Iceberg REST catalog *without* the Nessie SQL extension: without the extension, the REST client caches its `prefix` (branch selector) at first use, and you'd have to register a second catalog to hit a different ref. The Nessie SQL extension makes both `USE REFERENCE main` and `USE REFERENCE cat3_dev` first-class Spark SQL against the same catalog handle.

In [2]:
# `FROM main` is required by the parser — the extension has no implicit source.
r = spark.sql(f"CREATE BRANCH {DEV_BRANCH} IN nessie_catalog FROM main").collect()
print(f"created branch: {r[0].asDict()}")

# Switch this catalog handle onto the new branch.
spark.sql(f"USE REFERENCE {DEV_BRANCH} IN nessie_catalog").collect()
print(f"active ref now: {DEV_BRANCH}")

created branch: {'refType': 'Branch', 'name': 'cat3_dev', 'hash': '3dfcefab27e56d3d1b97a16eab8d13048bf67efb24d8ca9ba0670e5b1c6e0320'}
active ref now: cat3_dev


## 2. Do the work on `dev` — create + populate a table

This is the *rehearsal* step. In real ETL you'd run your migration, backfill, or destructive schema change here — whatever it is, `main` is untouched while you experiment.

> The table only exists on `cat3_dev` right now. That's the whole point of branch isolation.

In [3]:
spark.sql(f"""
CREATE TABLE {MAIN_TABLE} (
    order_id BIGINT,
    customer STRING,
    amount   DOUBLE,
    status   STRING
) USING iceberg
""").collect()

spark.sql(f"""
INSERT INTO {MAIN_TABLE} VALUES
    (1, 'alice', 50.0,  'PAID'),
    (2, 'bob',   75.0,  'NEW'),
    (3, 'carol', 120.0, 'PAID'),
    (4, 'dave',   30.0, 'NEW'),
    (5, 'eve',   200.0, 'PAID')
""").collect()

n_dev = spark.sql(f"SELECT COUNT(*) FROM {MAIN_TABLE}").collect()[0][0]
print(f"on {DEV_BRANCH}: {MAIN_TABLE} has {n_dev} rows")
spark.sql(f"SELECT * FROM {MAIN_TABLE} ORDER BY order_id").show(truncate=False)

on cat3_dev: nessie_catalog.staging.orders has 5 rows
+--------+--------+------+------+
|order_id|customer|amount|status|
+--------+--------+------+------+
|1       |alice   |50.0  |PAID  |
|2       |bob     |75.0  |NEW   |
|3       |carol   |120.0 |PAID  |
|4       |dave    |30.0  |NEW   |
|5       |eve     |200.0 |PAID  |
+--------+--------+------+------+



## 3. Detect — prove isolation from `main`

Switch this catalog handle back to `main`. The table shouldn't exist there — we never created it on `main`, only on `cat3_dev`. Query it and we expect a **`TABLE_OR_VIEW_NOT_FOUND`** analysis error.

That failure is the *proof*: at the metadata layer, branches are truly separate namespaces of state.

In [4]:
spark.sql("USE REFERENCE main IN nessie_catalog").collect()
print("active ref now: main")

isolated = False
try:
    n_main = spark.sql(f"SELECT COUNT(*) FROM {MAIN_TABLE}").collect()[0][0]
    print(f"UNEXPECTED: main sees {n_main} rows — isolation broke!")
except Exception as e:
    err = str(e)
    if "TABLE_OR_VIEW_NOT_FOUND" in err or "cannot be found" in err:
        isolated = True
        print("main does NOT see the table — isolation proven.")
        print(f"    (Spark reports: TABLE_OR_VIEW_NOT_FOUND for {MAIN_TABLE})")
    else:
        raise

assert isolated, "main saw the dev-only table — branch isolation is broken"
print("\n   dev wrote a table + 5 rows; main didn't see them. Same catalog, two histories.")

{"ts": "2026-07-21 10:27:00.537", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `nessie_catalog`.`staging`.`orders` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 21;\n'Aggregate [unresolvedalias(count(1))]\n+- 'UnresolvedRelation [nessie_catalog, staging, orders], [], false\n\n\nJVM stacktrace:\norg.apache.spark.sql.catalyst.ExtendedAnalysisException\n\tat org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.tableNotFound(package.scala:91)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$2(CheckAnalysis.scala:306)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$2$adapted(C

active ref now: main
main does NOT see the table — isolation proven.
    (Spark reports: TABLE_OR_VIEW_NOT_FOUND for nessie_catalog.staging.orders)

   dev wrote a table + 5 rows; main didn't see them. Same catalog, two histories.


## 4. Merge `dev` into `main` — one atomic commit

`MERGE BRANCH dev INTO main` is a single commit against Nessie's ref store. Downstream readers see main advance in one instant — there's no half-merged state, no window where readers get inconsistent metadata.

> Nessie's default merge is **not** a fast-forward — it adds an explicit merge commit on `main` that names both parent histories. Pass `defaultMergeType=FORCE` on the REST call (or the SQL extension's `FAST_FORWARD ONLY`) if you want the git-`ff-only` behavior.

In [5]:
r = spark.sql(f"MERGE BRANCH {DEV_BRANCH} INTO main IN nessie_catalog").collect()
print(f"merge result: {r[0].asDict()}")

merge result: {'name': 'main', 'hash': 'a73fa2002d82844fc72ebb437bb9ee7062759fb726f9ed1488d0770870a2e431'}


## 5. Prove — rows now visible on `main`

Same fully-qualified name, same catalog handle. Only the ref moved. This is the assertion the whole lesson has been building toward.

In [6]:
n_after = spark.sql(f"SELECT COUNT(*) FROM {MAIN_TABLE}").collect()[0][0]
print(f"main after merge: {n_after} rows")
spark.sql(f"SELECT * FROM {MAIN_TABLE} ORDER BY order_id").show(truncate=False)

assert n_after == 5, f"expected 5 rows on main after merge, got {n_after}"

# Look at the Iceberg snapshot lineage on main — there's one snapshot (from dev's insert),
# reachable now that main's ref has been advanced. Iceberg's per-table history and
# Nessie's cross-table commit graph coexist — this table's snapshot came in via merge.
print("\nsnapshots visible on main for this table:")
spark.sql(f"SELECT snapshot_id, operation FROM {MAIN_TABLE}.snapshots ORDER BY committed_at").show(truncate=False)

main after merge: 5 rows
+--------+--------+------+------+
|order_id|customer|amount|status|
+--------+--------+------+------+
|1       |alice   |50.0  |PAID  |
|2       |bob     |75.0  |NEW   |
|3       |carol   |120.0 |PAID  |
|4       |dave    |30.0  |NEW   |
|5       |eve     |200.0 |PAID  |
+--------+--------+------+------+


snapshots visible on main for this table:
+------------------+---------+
|snapshot_id       |operation|
+------------------+---------+
|191461471148889420|append   |
+------------------+---------+



## What you just saw

- **Branches are named pointers into a shared commit graph** — not copies of the warehouse. Forking `cat3_dev` from `main` cost one row in Nessie's ref store; the physical data files (Parquet) sit on S3 and would be shared between refs whenever histories overlap.
- **Isolation is a metadata property.** `main` didn't see the table because its ref didn't include the commit that created it — the S3 files were there the whole time; there was just no pointer path to them from `main`.
- **`MERGE` is one atomic commit** against Nessie's ref store. There is no window where `main` has "half" of the merge.
- **Same catalog handle, both refs.** With the Nessie Spark SQL extension, one `nessie_catalog` in `spark-defaults.conf` handles both `main` and `cat3_dev` via `USE REFERENCE`. No shadow catalog, no config reload.

### Try it yourself
- Run this notebook twice — the reset step at the top drops the table and any leftover branch, so it's fully idempotent.
- Create *two* dev branches from the same `main`, mutate them independently, and try `MERGE`'ing them in sequence. That's the multi-developer story.
- Open [`../catalog_api_playground.ipynb`](../catalog_api_playground.ipynb) and reproduce the same branch/merge via **raw Nessie REST** — the SQL and REST paths write to the same commit graph.

**Next:** [CAT-5 — federated ETL: Nessie staging → Glue marts](../cat5_federation.ipynb) puts branch-friendly staging + governed marts in one Spark job.

## Teardown

Drop the table on `main` (which now owns the merged rows) and delete the `cat3_dev` branch — leave the catalog clean for peer modules. `make clean` clears MiniStack + Nessie's RocksDB for a fully fresh start.

In [7]:
spark.sql(f"DROP TABLE IF EXISTS {MAIN_TABLE}").collect()
try:
    spark.sql(f"DROP BRANCH {DEV_BRANCH} IN nessie_catalog").collect()
    print(f"dropped {MAIN_TABLE} and branch {DEV_BRANCH}")
except Exception as e:
    print(f"branch cleanup tolerated: {type(e).__name__}")

dropped nessie_catalog.staging.orders and branch cat3_dev
